# Week 5 — Decision Tree and Random Forest

This notebook focuses on three tasks:

1. Train **Decision Tree** and **Random Forest** regressors.
2. Compare their **April 2026 test R²** with the Linear Regression baseline.
3. Document each model's main strengths and weaknesses.

All three models use the same rows, features, and preprocessing. **R²** measures how much of the variation in sale prices the model explains; higher values are better.

In [ ]:
# Import the tools used to prepare data, train models, and plot results.
from pathlib import Path
import json
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, TargetEncoder
from sklearn.tree import DecisionTreeRegressor, plot_tree

RANDOM_STATE = 42

## 1. Load the data

Notebook 02 created chronological splits: training data through February 2026, March validation data, and April test data.

In [ ]:
# Resolve processed and raw-data paths from either the repository root or notebooks folder.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
configured_raw_data_dir = os.getenv("CRMLS_SOLD_DATA_DIR")
RAW_DATA_DIR = (
    Path(configured_raw_data_dir).expanduser()
    if configured_raw_data_dir
    else REPO_ROOT / "IDX_private_data" / "CRMLSSold"
)
assert RAW_DATA_DIR.exists(), f"Raw data folder not found: {RAW_DATA_DIR}"

In [ ]:
# Load notebook 02's training, March validation, and April test splits.
split_paths = {
    "Training": PROCESSED_DIR / "train_clean.csv",
    "Validation": PROCESSED_DIR / "validation_clean.csv",
    "Test": PROCESSED_DIR / "test_clean.csv",
}
missing_files = [str(path) for path in split_paths.values() if not path.exists()]
assert not missing_files, f"Run notebook 02 first. Missing: {missing_files}"

splits = {
    name: pd.read_csv(path, low_memory=False)
    for name, path in split_paths.items()
}

In [ ]:
# Verify each split's date range and row count.
split_summary = pd.DataFrame([
    [name, data["SourceMonth"].min(), data["SourceMonth"].max(), len(data)]
    for name, data in splits.items()
], columns=["Split", "First month", "Last month", "Rows"]).set_index("Split")

display(split_summary)

## 2. Prepare the same features for every model

The models use property size, room counts, location, age, sale month, amenities, missing-value indicators, and geographic categories. Using identical inputs makes the R² comparison fair.

In [ ]:
# Group size, count, location, age, season, amenity, and category features by preprocessing need.
TARGET = "ClosePrice"

AREA_FEATURES = ["LivingArea", "LotSizeSquareFeet"]
PROPERTY_COUNT_FEATURES = [
    "BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces", "ParkingTotal",
]
COORDINATE_FEATURES = ["Latitude", "Longitude"]
AGE_FEATURES = ["PropertyAgeAtSale"]
TEMPORAL_FEATURES = ["SaleMonthSin", "SaleMonthCos"]
BINARY_FEATURES = [
    "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
    "PoolPrivateYN", "ViewYN",
]
CATEGORICAL_FEATURES = ["City", "CountyOrParish", "PostalCode"]

In [ ]:
# Add missingness flags and build the shared 33-feature model schema.
MISSINGNESS_BASE_FEATURES = (
    AREA_FEATURES + PROPERTY_COUNT_FEATURES
    + COORDINATE_FEATURES + AGE_FEATURES + BINARY_FEATURES
)
MISSING_INDICATOR_FEATURES = [
    f"{column}_Missing" for column in MISSINGNESS_BASE_FEATURES
]
MODEL_FEATURES = (
    AREA_FEATURES + PROPERTY_COUNT_FEATURES + COORDINATE_FEATURES
    + AGE_FEATURES + TEMPORAL_FEATURES + BINARY_FEATURES
    + MISSING_INDICATOR_FEATURES + CATEGORICAL_FEATURES
)

print(f"Configured {len(MODEL_FEATURES)} model features.")

In [ ]:
# List predictors missing from notebook 02's exports for recovery from raw files.
RAW_EXTRA_FEATURES = [
    "LotSizeSquareFeet", "LotSizeAcres",
    "BedroomsTotal", "BathroomsTotalInteger",
    "GarageSpaces", "ParkingTotal",
    "Latitude", "Longitude", "YearBuilt",
    "City", "CountyOrParish", "PostalCode",
]

In [ ]:
# Map each YYYY-MM source month to its raw CSV and verify complete coverage.
def month_from_filename(path):
    stamp = path.stem.replace("CRMLSSold", "").replace("_filled", "")
    return f"{stamp[:4]}-{stamp[4:6]}"

raw_files_by_month = {
    month_from_filename(path): path
    for path in RAW_DATA_DIR.glob("CRMLSSold*.csv")
}
required_months = sorted({
    str(month)
    for data in splits.values()
    for month in data["SourceMonth"].unique()
})
missing_months = sorted(set(required_months) - set(raw_files_by_month))
assert not missing_months, f"Missing raw months: {missing_months}"

In [ ]:
# Load raw predictors, standardize ListingKey, and remove duplicate key-month rows.
raw_months = []
raw_columns = ["ListingKey"] + RAW_EXTRA_FEATURES

for month in required_months:
    monthly = pd.read_csv(
        raw_files_by_month[month],
        usecols=raw_columns,
        low_memory=False,
    )
    monthly["SourceMonth"] = month
    monthly["ListingKey"] = (
        monthly["ListingKey"].astype("string")
        .str.replace(r"\.0$", "", regex=True)
    )
    monthly = monthly.drop_duplicates(["SourceMonth", "ListingKey"])
    raw_months.append(monthly)

raw_features = pd.concat(raw_months, ignore_index=True)

In [ ]:
# Join raw predictors by month and ListingKey without dropping or multiplying cleaned rows.
for name, data in splits.items():
    data = data.copy()
    data["SourceMonth"] = data["SourceMonth"].astype(str)
    data["ListingKey"] = (
        data["ListingKey"].astype("string")
        .str.replace(r"\.0$", "", regex=True)
    )
    enriched = data.merge(
        raw_features,
        on=["SourceMonth", "ListingKey"],
        how="left",
        validate="many_to_one",
        indicator=True,
    )
    assert len(enriched) == len(data)
    assert enriched["_merge"].eq("both").all()
    splits[name] = enriched.drop(columns="_merge")

print(f"Enriched {sum(map(len, splits.values())):,} rows.")

In [ ]:
# Coerce numeric types, recover lot area, and mark implausible counts or coordinates as missing.
COUNT_BOUNDS = {
    "BedroomsTotal": (0, 20),
    "BathroomsTotalInteger": (0, 20),
    "GarageSpaces": (0, 10),
    "ParkingTotal": (0, 20),
}
NUMERIC_FEATURES = (
    AREA_FEATURES + PROPERTY_COUNT_FEATURES
    + COORDINATE_FEATURES + ["LotSizeAcres", "YearBuilt"]
)

# Clean numeric values before deriving the shared model features.
def clean_numeric_features(data):
    cleaned = data.copy()
    cleaned[NUMERIC_FEATURES] = cleaned[NUMERIC_FEATURES].apply(
        pd.to_numeric, errors="coerce"
    )
    cleaned[TARGET] = pd.to_numeric(cleaned[TARGET], errors="coerce")

    cleaned.loc[cleaned["LivingArea"].le(0), "LivingArea"] = np.nan
    cleaned.loc[
        cleaned["LotSizeSquareFeet"].le(0), "LotSizeSquareFeet"
    ] = np.nan

    recover_lot = (
        cleaned["LotSizeSquareFeet"].isna()
        & cleaned["LotSizeAcres"].gt(0)
    )
    cleaned.loc[recover_lot, "LotSizeSquareFeet"] = (
        cleaned.loc[recover_lot, "LotSizeAcres"] * 43_560
    )

    for column, (lower, upper) in COUNT_BOUNDS.items():
        invalid = cleaned[column].notna() & ~cleaned[column].between(lower, upper)
        cleaned.loc[invalid, column] = np.nan

    cleaned.loc[
        ~cleaned["Latitude"].between(32.5, 42.1), "Latitude"
    ] = np.nan
    cleaned.loc[
        ~cleaned["Longitude"].between(-124.5, -114.0), "Longitude"
    ] = np.nan
    return cleaned

In [ ]:
# Derive valid property age and cyclical sale-month sine/cosine features.
def add_age_and_season(data):
    prepared = data.copy()

    close_year = pd.to_datetime(
        prepared["CloseDate"], errors="coerce"
    ).dt.year
    prepared["PropertyAgeAtSale"] = close_year - prepared["YearBuilt"]
    prepared.loc[
        ~prepared["PropertyAgeAtSale"].between(0, 250),
        "PropertyAgeAtSale",
    ] = np.nan

    sale_month = pd.PeriodIndex(prepared["SourceMonth"], freq="M").month
    prepared["SaleMonthSin"] = np.sin(2 * np.pi * sale_month / 12)
    prepared["SaleMonthCos"] = np.cos(2 * np.pi * sale_month / 12)
    return prepared

In [ ]:
# Normalize binary and category fields, then create pre-imputation missingness flags.
def finish_features(data):
    prepared = data.copy()

    for column in BINARY_FEATURES:
        prepared[column] = pd.to_numeric(
            prepared[column], errors="coerce"
        )

    for column in MISSINGNESS_BASE_FEATURES:
        prepared[f"{column}_Missing"] = (
            prepared[column].isna().astype(float)
        )

    for column in ["City", "CountyOrParish"]:
        prepared[column] = (
            prepared[column].astype("string").str.strip()
            .replace("", pd.NA).fillna("Unknown")
        )

    prepared["PostalCode"] = (
        prepared["PostalCode"].astype("string")
        .str.extract(r"(\d{5})", expand=False)
        .fillna("Unknown")
    )
    return prepared

In [ ]:
# Apply identical fixed feature preparation to training, validation, and test.
def prepare_features(data):
    data = clean_numeric_features(data)
    data = add_age_and_season(data)
    return finish_features(data)

splits = {
    name: prepare_features(data)
    for name, data in splits.items()
}

In [ ]:
# Verify required columns, positive targets, and valid binary values in every split.
for name, data in splits.items():
    required = MODEL_FEATURES + [TARGET, "SourceMonth"]
    missing = sorted(set(required) - set(data.columns))
    assert not missing, f"{name} is missing: {missing}"
    assert data[TARGET].notna().all()
    assert data[TARGET].gt(0).all()

    for column in BINARY_FEATURES:
        valid_values = set(data[column].dropna().unique())
        assert valid_values.issubset({0.0, 1.0})

print("Feature checks passed.")

In [ ]:
# Save Notebook 04's prepared rows and schema for Notebook 05.
handoff_paths = {
    "Training": PROCESSED_DIR / "notebook04_training_features.csv.gz",
    "Validation": PROCESSED_DIR / "notebook04_validation_features.csv.gz",
    "Test": PROCESSED_DIR / "notebook04_test_features.csv.gz",
}
handoff_columns = ["ListingKey", "SourceMonth", TARGET] + MODEL_FEATURES
for name, path in handoff_paths.items():
    splits[name][handoff_columns].to_csv(
        path, index=False, compression="gzip"
    )

feature_schema = {
    "target": TARGET,
    "area": AREA_FEATURES,
    "property_counts": PROPERTY_COUNT_FEATURES,
    "coordinates": COORDINATE_FEATURES,
    "age": AGE_FEATURES,
    "temporal": TEMPORAL_FEATURES,
    "binary": BINARY_FEATURES,
    "missing_indicators": MISSING_INDICATOR_FEATURES,
    "categorical": CATEGORICAL_FEATURES,
    "model_features": MODEL_FEATURES,
}
with (PROCESSED_DIR / "notebook04_feature_schema.json").open("w") as file:
    json.dump(feature_schema, file, indent=2)

print("Saved prepared Notebook 04 features for Notebook 05.")

## 3. Build shared preprocessing

The pipeline learns missing-value replacements, scaling, and category encodings from the training rows. The same fitted transformations are then applied to the test rows.

In [ ]:
# Impute coordinates with training ZIP, county, then statewide medians.
class GeographicCoordinateImputer(BaseEstimator, TransformerMixin):
    # Learn coordinate medians from the development rows.
    def fit(self, X, y=None):
        self.zip_medians_ = (
            X.groupby("PostalCode")[COORDINATE_FEATURES].median()
        )
        self.county_medians_ = (
            X.groupby("CountyOrParish")[COORDINATE_FEATURES].median()
        )
        self.global_medians_ = X[COORDINATE_FEATURES].median()
        assert self.global_medians_.notna().all()
        return self

    # Apply the learned medians to new rows.
    def transform(self, X):
        result = X.copy()
        for column in COORDINATE_FEATURES:
            result[column] = result[column].fillna(
                result["PostalCode"].map(self.zip_medians_[column])
            )
            result[column] = result[column].fillna(
                result["CountyOrParish"].map(
                    self.county_medians_[column]
                )
            )
            result[column] = result[column].fillna(
                self.global_medians_[column]
            )
        return result

In [ ]:
# Define imputation, transformation, scaling, and target encoding by feature group.
area_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log1p", FunctionTransformer(
        np.log1p, feature_names_out="one-to-one"
    )),
    ("scaler", StandardScaler()),
])
count_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("scaler", StandardScaler()),
])
age_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant", fill_value="Unknown"
    )),
    ("encoder", TargetEncoder(
        target_type="continuous",
        cv=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )),
])

In [ ]:
# Combine feature-specific preprocessing into one shared 33-column transformer.
PREPROCESSOR = ColumnTransformer([
    ("area", area_pipeline, AREA_FEATURES),
    ("counts", count_pipeline, PROPERTY_COUNT_FEATURES),
    ("coordinates", StandardScaler(), COORDINATE_FEATURES),
    ("age", age_pipeline, AGE_FEATURES),
    ("temporal", "passthrough", TEMPORAL_FEATURES),
    ("binary", binary_pipeline, BINARY_FEATURES),
    ("missing", "passthrough", MISSING_INDICATOR_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
], verbose_feature_names_out=False)

In [ ]:
# Attach any regressor to fresh coordinate and feature preprocessing steps.
def build_model_pipeline(regressor):
    return Pipeline([
        ("coordinate_imputation", GeographicCoordinateImputer()),
        ("preprocessor", clone(PREPROCESSOR)),
        ("regressor", regressor),
    ])

## 4. Train the baseline and two tree models

March validation rows are added to the training rows before final fitting. April remains separate and is used only for the final test comparison.

- **Linear Regression** is the baseline.
- **Decision Tree** makes one prediction path through a series of yes/no splits.
- **Random Forest** averages 30 different trees to produce a more stable prediction.

In [ ]:
# Combine training and validation rows, then keep April as the final test set.
development = pd.concat(
    [splits["Training"], splits["Validation"]],
    ignore_index=True,
)
test = splits["Test"].copy()

X_development = development[MODEL_FEATURES]
y_development_log = np.log1p(development[TARGET])
X_test = test[MODEL_FEATURES]
y_test = test[TARGET]

print(f"Development rows: {len(development):,}")
print(f"April test rows: {len(test):,}")

In [ ]:
# Define the baseline and regularized tree models before viewing test results.
models = {
    "Linear Regression": LinearRegression(n_jobs=-1),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=18,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=30,
        max_depth=18,
        min_samples_leaf=10,
        max_features=0.7,
        max_samples=0.7,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

In [ ]:
# Fit each model on the same data and calculate its April test R².
fitted_models = {}
test_predictions = {}
result_rows = []

for model_name, regressor in models.items():
    pipeline = build_model_pipeline(regressor)
    pipeline.fit(X_development, y_development_log)

    predictions = np.expm1(pipeline.predict(X_test))
    fitted_models[model_name] = pipeline
    test_predictions[model_name] = predictions
    result_rows.append({
        "Model": model_name,
        "Test R²": r2_score(y_test, predictions),
        "Test MAE": mean_absolute_error(y_test, predictions),
        "Test RMSE": mean_squared_error(y_test, predictions) ** 0.5,
    })

## 5. Compare test R² with the baseline

A positive change means the model explains more test-price variation than Linear Regression.

In [ ]:
# Report each test R² and its change from the Linear Regression baseline.
model_order = ["Linear Regression", "Decision Tree", "Random Forest"]
results = pd.DataFrame(result_rows).set_index("Model").loc[model_order]
baseline_r2 = results.loc["Linear Regression", "Test R²"]
results["R² change vs baseline"] = results["Test R²"] - baseline_r2
results.reset_index().to_csv(
    PROCESSED_DIR / "notebook04_model_results.csv", index=False
)

display(results.style.format({
    "Test R²": "{:.4f}",
    "Test MAE": "${:,.0f}",
    "Test RMSE": "${:,.0f}",
    "R² change vs baseline": "{:+.4f}",
}))

In [ ]:
colors = ["#7f8c8d", "#f39c12", "#2e86de"]
scores = results["Test R²"].to_numpy()
changes = results["R² change vs baseline"].to_numpy()
y_positions = np.arange(len(model_order))

fig, ax = plt.subplots(figsize=(9, 4.5))

bars = ax.barh(
    y_positions,
    scores,
    color=colors,
    height=0.6,
)

# Show the Linear Regression baseline.
ax.axvline(
    baseline_r2,
    color="#555555",
    linestyle="--",
    linewidth=1.5,
    label=f"Linear Regression baseline ({baseline_r2:.4f})",
)

# Add score and change labels.
for bar, score, change in zip(bars, scores, changes):
    label = f"{score:.4f}"
    if not np.isclose(change, 0):
        label += f"  ({change:+.4f})"

    ax.text(
        score + 0.003,
        bar.get_y() + bar.get_height() / 2,
        label,
        va="center",
        fontsize=9,
    )

ax.set(
    title="April 2026 test R² by model",
    xlabel="Test R² (higher is better)",
    yticks=y_positions,
    yticklabels=model_order,
    xlim=(max(0, scores.min() - 0.03), min(1, scores.max() + 0.06)),
)

ax.invert_yaxis()
ax.grid(axis="x", alpha=0.25)
ax.legend(frameon=False)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis="y", length=0)
fig.tight_layout()
plt.show()

## 6. See how Decision Tree and Random Forest differ

The left diagram is the single Decision Tree. The right diagram is only one of the Random Forest's 30 trees. The forest repeats this process across all 30 trees and averages their predictions.

In [ ]:
# Display the first three levels so the single-tree and forest structures are readable.
decision_pipeline = fitted_models["Decision Tree"]
forest_pipeline = fitted_models["Random Forest"]
feature_names = (
    decision_pipeline.named_steps["preprocessor"]
    .get_feature_names_out()
)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

plot_tree(
    decision_pipeline.named_steps["regressor"],
    max_depth=2,
    feature_names=feature_names,
    filled=True,
    rounded=True,
    impurity=False,
    proportion=True,
    precision=2,
    fontsize=7,
    ax=axes[0],
)
axes[0].set_title("Decision Tree: one tree makes the prediction")

plot_tree(
    forest_pipeline.named_steps["regressor"].estimators_[0],
    max_depth=2,
    feature_names=feature_names,
    filled=True,
    rounded=True,
    impurity=False,
    proportion=True,
    precision=2,
    fontsize=7,
    ax=axes[1],
)
axes[1].set_title("Random Forest: one of 30 trees")

fig.tight_layout()
plt.show()

## 7. Model behavior: strengths and weaknesses

| Model | Strengths | Weaknesses |
|---|---|---|
| Linear Regression | Fast and easy to interpret; provides a clear baseline. | Assumes mostly linear relationships, so it can miss complex price patterns. |
| Decision Tree | Learns nonlinear thresholds and feature interactions; its decision path can be inspected. | One tree can change greatly with small data changes and can overfit. |
| Random Forest | Averaging many trees reduces overfitting and usually improves prediction stability. | Slower and harder to interpret than one tree or a linear model. |

## Conclusion

On the April 2026 test set, the Decision Tree and Random Forest both achieved higher R² than the Linear Regression baseline. Random Forest performed best because averaging multiple trees reduced the instability of a single Decision Tree.